# 02 – WAR Model (v0.1)

**Goal:** a first working WAR number for every skater, built from three components:

| Component | What it measures |
|---|---|
| **EV offense** | Shot quality created + primary assists, above replacement |
| **EV defense** | How much better the team defends with him on the ice vs. on the bench, above replacement |
| **Penalties** | Penalties drawn minus taken, converted to goals |

Each component is measured in **goals above replacement (GAR)**, added up, then converted to **wins (WAR)**.

**Not in v0.1 (on purpose):** power play, penalty kill, faceoffs, finishing talent. Those come in v0.2. Start simple, check it works, then add complexity.

Requires: `01_data_prep.ipynb` has been run (it creates `data/skaters_clean.csv` and `data/replacement_rates.csv`).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

## 2. Load data
`skaters_clean.csv` is the table from notebook 01. Defense needs two columns we didn't keep there, **time on bench** and **team xG against while he's on the bench**, so we pull those from the raw file and join them by `playerId`.

In [ ]:
players = pd.read_csv(DATA / "skaters_clean.csv")
replacement_01 = pd.read_csv(DATA / "replacement_rates.csv", index_col=0)

raw = pd.read_csv(DATA / "skaters25-26.csv")
bench = (raw.loc[raw["situation"] == "5on5", ["playerId", "timeOnBench", "OffIce_A_xGoals"]]
            .rename(columns={"timeOnBench": "ev_bench_sec", "OffIce_A_xGoals": "ev_OffIce_A_xGoals"}))

players = players.merge(bench, on="playerId", how="left")
players["ev_bench_toi"] = players["ev_bench_sec"] / 60

print(players.shape)
replacement_01

## 3. Model assumptions
Every model has assumptions. Writing them down as named constants makes them easy to find, explain, and change later.

| Constant | Value | Why |
|---|---|---|
| `GOALS_PER_WIN` | 6.0 | In the NHL, roughly 6 extra goals of goal differential ≈ 1 extra win. Public models use ~5.5–6. |
| `PENALTY_GOAL_VALUE` | 0.17 | A penalty gives the other team a power play, worth roughly 0.15–0.2 goals on average. |
| `A1_WEIGHT` | 0.5 | A primary assist gets half credit for the goal. A judgment call, flagged as a limitation. |

In [ ]:
GOALS_PER_WIN = 6.0
PENALTY_GOAL_VALUE = 0.17
A1_WEIGHT = 0.5

## 4. Replacement levels for defense and penalties
Offense replacement rates come from notebook 01. Defense and penalties need their own, built the same way: **pooled** over the replacement group (non-regulars with 100+ EV minutes).

**Defense metric (`ev_rel_xga_60`):** team xG allowed per 60 when he's on the **bench**, minus team xG allowed per 60 when he's on the **ice**.
- Positive = the team gives up fewer chances with him out there.
- Comparing to his own team's bench numbers is what reduces team-quality bias (a mediocre defenseman on a great team doesn't get free credit).

In [ ]:
players["ev_xga_on_60"]  = np.where(players["ev_toi"] > 0,
                                    players["ev_OnIce_A_xGoals"] / players["ev_toi"] * 60, np.nan)
players["ev_xga_off_60"] = np.where(players["ev_bench_toi"] > 0,
                                    players["ev_OffIce_A_xGoals"] / players["ev_bench_toi"] * 60, np.nan)
players["ev_rel_xga_60"] = (players["ev_xga_off_60"] - players["ev_xga_on_60"]).fillna(0)

pool = players[(~players["is_regular"]) & (players["ev_toi"] >= 100)]

def pooled_def_and_pen(g):
    return pd.Series({
        "rel_xga_60": (g["ev_rel_xga_60"] * g["ev_toi"]).sum() / g["ev_toi"].sum(),
        "pen_diff_60": g["pen_diff"].sum() / g["all_toi"].sum() * 60,
    })

replacement = replacement_01.join(pool.groupby("pos_group").apply(pooled_def_and_pen, include_groups=False))
replacement.round(3)

## 5. Component 1: Even-strength offense
For each player: what he produced minus what a replacement player would have produced **in the same minutes**.

`GAR = (his xG − replacement xG rate × his minutes) + 0.5 × (his primary assists − replacement A1 rate × his minutes)`

Note: this uses **expected** goals (shot quality), not actual goals, so finishing luck is ignored for now.

In [ ]:
def repl(stat):
    """Replacement rate for each player's position group."""
    return players["pos_group"].map(replacement[stat])

ev_hours = players["ev_toi"] / 60

players["off_gar"] = (
    (players["ev_I_F_xGoals"] - repl("ixg_60") * ev_hours)
    + A1_WEIGHT * (players["ev_I_F_primaryAssists"] - repl("a1_60") * ev_hours)
)

players.sort_values("off_gar", ascending=False)[["name", "team", "position", "ev_toi", "off_gar"]].head(10).round(2)

## 6. Component 2: Even-strength defense
How many expected goals against he saved compared to a replacement player in the same minutes, measured relative to his own team.

In [ ]:
players["def_gar"] = (players["ev_rel_xga_60"] - repl("rel_xga_60")) * ev_hours

players.sort_values("def_gar", ascending=False)[["name", "team", "position", "ev_toi", "def_gar"]].head(10).round(2)

## 7. Component 3: Penalties
Net penalties drawn above replacement, times what each penalty is worth in goals.

In [ ]:
all_hours = players["all_toi"] / 60

players["pen_gar"] = PENALTY_GOAL_VALUE * (players["pen_diff"] - repl("pen_diff_60") * all_hours)

players.sort_values("pen_gar", ascending=False)[["name", "team", "position", "pen_diff", "pen_gar"]].head(10).round(2)

## 8. Total: goals → wins
Add the three components (GAR), then divide by goals per win.

In [ ]:
players["gar"] = players["off_gar"] + players["def_gar"] + players["pen_gar"]
players["war"] = players["gar"] / GOALS_PER_WIN

print(players["war"].describe().round(2))

## 9. Leaderboard
Top 20 skaters by WAR (v0.1).

In [ ]:
LEADER_COLS = ["name", "team", "position", "games_played", "ev_toi", "off_gar", "def_gar", "pen_gar", "war"]

leaders = players.sort_values("war", ascending=False)[LEADER_COLS].head(20).reset_index(drop=True)
leaders.index = leaders.index + 1
leaders.round(2)

### Top 5 by position
Forwards and defensemen side by side.

In [ ]:
for grp, label in [("F", "Forwards"), ("D", "Defensemen")]:
    print(label)
    print(players[players["pos_group"] == grp]
          .sort_values("war", ascending=False)[["name", "team", "war"]]
          .head(5).round(2).to_string(index=False))
    print()

## 10. Chart: what drives the top 15
Each bar is split into the three components, so you can see *why* a player ranks where he does. Negative components extend left of zero. The chart is saved to `images/` so it can go in the README.

In [ ]:
COLORS = {"off_gar": "#2a78d6", "def_gar": "#eb6834", "pen_gar": "#1baf7a"}   # blue / orange / aqua
LABELS = {"off_gar": "EV offense", "def_gar": "EV defense", "pen_gar": "Penalties"}
SURFACE, INK, MUTED = "#fcfcfb", "#0b0b0b", "#52514e"

top = players.sort_values("war", ascending=False).head(15).iloc[::-1]   # reversed so #1 is on top
comp_wins = top[list(COLORS)] / GOALS_PER_WIN                            # components in wins

fig, ax = plt.subplots(figsize=(9, 7), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

pos_left = np.zeros(len(top))
neg_left = np.zeros(len(top))
for col, color in COLORS.items():
    vals = comp_wins[col].to_numpy()
    left = np.where(vals >= 0, pos_left, neg_left)
    ax.barh(top["name"], vals, left=left, color=color, height=0.62,
            edgecolor=SURFACE, linewidth=2, label=LABELS[col])
    pos_left += np.where(vals >= 0, vals, 0)
    neg_left += np.where(vals < 0, vals, 0)

for y, (total, end) in enumerate(zip(top["war"], pos_left)):
    ax.text(end + 0.05, y, f"{total:.1f}", va="center", fontsize=9, color=INK)

ax.axvline(0, color=MUTED, linewidth=1)
ax.set_xlabel("Wins above replacement", color=MUTED)
ax.set_title("Top 15 skaters by WAR, 2025-26 (v0.1: EV + penalties)", loc="left", color=INK, fontsize=12)
ax.tick_params(colors=MUTED, labelsize=9)
ax.grid(axis="x", color="#e6e5e1", linewidth=0.8)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=MUTED)
fig.text(0.01, 0.01, "Data: MoneyPuck.com", fontsize=8, color=MUTED)

plt.tight_layout()
fig.savefig(IMAGES / "war_top15_v01.png", dpi=150, facecolor=SURFACE)
plt.show()

## 11. Sanity check: where are the superstars?
A good analyst checks the results against what they know. Some of the league's best players should be near the top. Where are they?

In [ ]:
players["war_rank"] = players["war"].rank(ascending=False, method="min").astype(int)

STARS = ["Connor McDavid", "Nathan MacKinnon", "Nikita Kucherov", "Leon Draisaitl",
         "Auston Matthews", "Cale Makar", "Quinn Hughes", "David Pastrnak"]

(players[players["name"].isin(STARS)]
    [["name", "team", "war_rank", "off_gar", "def_gar", "pen_gar", "war", "pp_toi"]]
    .sort_values("war_rank").round(2))

**Question for you before v0.2:** several of these stars rank lower than you'd expect. Look at the components and the `pp_toi` column. What's the model missing that would push them up?

## 12. Save

In [ ]:
players.to_csv(DATA / "war_v01.csv", index=False)
print("Saved", len(players), "players to data/war_v01.csv")

## Known limitations (v0.1)
- **No special teams or faceoffs**: power play specialists and elite centers are undervalued
- **Offense uses expected goals, not goals**: elite finishers don't get credit for finishing
- **Defense is noisy**: on-ice xG against depends on linemates and competition, not just the player
- **Assist weight (0.5) is a judgment call**, not derived from data
- **Traded players** are listed under one team, so their team-relative numbers are approximate

## Next: v0.2
Add power play, penalty kill, and faceoffs, credit finishing, and compare the leaderboard to a public WAR model.